# Report figures

Figures for the CS6101 report (ACM `sigconf`, two columns) and the Beamer slides. The notebook reads the episode logs only; it never calls the API.

| Output | Width | Content |
|---|---|---|
| `fig1_setup` | full width | The game tree with payoffs and information, and a notation key |
| `fig2_results` | one column | Capture without and with the band, rational client against LLM (slope chart) |
| `fig3_counters` | one column | Where the client counters, without and with the band |
| `fig4_evidence` | one column | How far the dealer's quote and the band each move the client's best guess, LLM against Bayesian |
| `fig5_heldout` | one column | Main against held-out: band effect on capture, bids and best guess moved towards the band's centre |

Each figure is saved as a vector PDF (for LaTeX) and a 300 dpi PNG to `report/figures/`. The last two sections print the numbers quoted in the text. Run all cells top to bottom.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# run from the project root so the project modules and logs are found
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import yaml
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy import stats

from disclosure_results import (load_episodes, per_scenario, bootstrap_ci, paired, capture,
                                benchmark)
from generator import make_scenario, simulate_disclosure
from benchmarks import band_posterior, prior_density_c
from rational import price_grid, dealer_best_quote

CAL = yaml.safe_load(open("config.yaml"))["calibration"]
G, D = CAL["g_points"], CAL["disclosure"]
S_BAND = float(np.sqrt(D["omega"] ** 2 / D["K"] + D["tau"] ** 2))      # band half-width, in g

OUT = ROOT / "report" / "figures"
OUT.mkdir(parents=True, exist_ok=True)

MAIN = load_episodes("logs/main_ornith1.5-35b.jsonl")
HELDOUT = load_episodes("logs/heldout_ornith1.5-35b.jsonl")
SEEDS_MAIN, SEEDS_HELD = range(1, 51), range(1001, 1021)
print(f"main episodes {len(MAIN)}, held-out episodes {len(HELDOUT)}, g = {G}")

### Style

Times to match the ACM template, 8 pt text, no titles inside figures (captions go in LaTeX), and a colour-blind-safe palette that also reads in greyscale.

In [ ]:
COL = 3.31          # ACM sigconf column width, inches
FULL = 7.0          # ACM sigconf text width, inches

NOBAND = "#595959"  # no band: dark grey
BAND = "#0072B2"    # band: blue (Okabe-Ito)
BAND_LIGHT = "#C6DBEF"
RATIONAL = "#000000"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral"],
    "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.direction": "out", "ytick.direction": "out",
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.handlelength": 1.5,
    "lines.linewidth": 1.0, "errorbar.capsize": 0,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,          # embed TrueType fonts (journals require it)
})


def panel(ax, letter, x=-0.16, y=1.04):
    ax.text(x, y, f"({letter})", transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom")


def save(fig, name):
    fig.savefig(OUT / f"{name}.pdf")
    fig.savefig(OUT / f"{name}.png")
    print("saved", OUT / f"{name}.pdf")

## Numbers behind the figures

In [ ]:
ARMS = [("U_noelicit", "no band, no questions"), ("B_noelicit", "band, no questions"),
        ("U", "no band, questions"), ("B", "band, questions")]

bench_main = benchmark(SEEDS_MAIN, G, D)
bench_held = benchmark(SEEDS_HELD, G, D)

def arm_ci(recs, cond, seeds):
    vals = per_scenario(recs, cond, capture, seeds)
    return bootstrap_ci(list(vals.values()))

def rational_ci(bench, which):
    return bootstrap_ci([getattr(b[which], "client_capture") for b in bench.values()])

def rational_gain(bench):
    return float(np.mean([b["B"].client_capture - b["U"].client_capture for b in bench.values()]))

capture_by_arm = {c: arm_ci(MAIN, c, SEEDS_MAIN) for c, _ in ARMS}
rational = {w: rational_ci(bench_main, w) for w in ("U", "B")}

effects = [   # (label, recs, hi, lo, seeds, bench)
    ("Main, no questions (H1)", MAIN, "B_noelicit", "U_noelicit", SEEDS_MAIN, bench_main),
    ("Main, belief questions", MAIN, "B", "U", SEEDS_MAIN, bench_main),
    ("Held-out, no questions", HELDOUT, "B_noelicit", "U_noelicit", SEEDS_HELD, bench_held),
    ("Held-out, belief questions", HELDOUT, "B", "U", SEEDS_HELD, bench_held),
]
effect_rows = []
for label, recs, hi, lo, seeds, bench in effects:
    p = paired(recs, hi, lo, capture, seeds)
    effect_rows.append((label, p["diff"], p["ci"], rational_gain(bench)))

for c, name in ARMS:
    m, (lo, hi) = capture_by_arm[c]
    print(f"{name:<24} capture {m:.3f}  [{lo:.3f}, {hi:.3f}]")
for w in ("U", "B"):
    m, (lo, hi) = rational[w]
    print(f"rational {w:<15} capture {m:.3f}  [{lo:.3f}, {hi:.3f}]")
print()
for label, d, (lo, hi), rg in effect_rows:
    print(f"{label:<28} LLM {d:+.3f} [{lo:+.3f}, {hi:+.3f}]   rational {rg:+.3f}")

## Figure 1: the game and its notation

Full width (`figure*`). The game tree, with what each outcome pays and what each side knows, and a notation key.

In [ ]:
import textwrap
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

# ---------------------------------------------------------------- Figure 1: the game and its notation
DEALER_C, CLIENT_C = "#D55E00", BAND           # dealer: vermilion; client: blue

fig = plt.figure(figsize=(FULL, 1.95))
ax = fig.add_axes([0.0, 0.0, 0.62, 1.0])
ax.set_xlim(0, 10); ax.set_ylim(0, 5.2); ax.axis("off")


def box(x, y, who, col, w=1.25, h=0.62):
    b = FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0,rounding_size=0.12",
                       fc="white", ec=col, lw=1.2, zorder=3)
    ax.add_patch(b)
    ax.text(x, y, who, ha="center", va="center", fontsize=7.5, color=col, fontweight="bold", zorder=4)
    return b


def edge(a, b, label, at, ha="center", pa=None, pb=None):
    ax.add_patch(FancyArrowPatch(a, b, patchA=pa, patchB=pb, arrowstyle="-|>", mutation_scale=7,
                                 lw=0.8, color="0.25", shrinkA=1, shrinkB=1, zorder=2))
    ax.text(*at, label, ha=ha, va="center", fontsize=7)


def leaf(x, y, outcome, pay):
    ax.text(x, y + 0.03, outcome, ha="left", va="bottom", fontsize=7, fontweight="bold")
    ax.text(x, y - 0.03, pay, ha="left", va="top", fontsize=6.6, color="0.25")


D1, C2, D3 = (0.75, 3.3), (3.35, 3.3), (5.75, 1.55)
b1 = box(*D1, "Dealer", DEALER_C)
b2 = box(*C2, "Client", CLIENT_C)
b3 = box(*D3, "Dealer", DEALER_C)
edge(D1, C2, "1. quotes $p$", at=(2.05, 3.55), pa=b1, pb=b2)
edge(C2, (6.9, 4.55), "2. accepts $p$", at=(4.9, 4.25), ha="right", pa=b2)
edge(C2, D3, "2. counters $q < p$", at=(4.35, 2.2), ha="right", pa=b2, pb=b3)
edge(D3, (7.55, 2.55), r"3. $q \geq c$: accepts", at=(6.55, 2.45), ha="right", pa=b3)
edge(D3, (7.55, 0.6), r"3. $q < c$: rejects", at=(6.55, 0.62), ha="right", pa=b3)
leaf(7.0, 4.55, "trade at $p$", "client gets $r-p$, dealer $p-c$")
leaf(7.65, 2.55, "trade at $q$", "client gets $r-q$, dealer $q-c$")
leaf(7.65, 0.6, "no trade", "both get 0")
ax.text(0.1, 1.15, "Dealer knows: $m$, its cost $c$, the prior on $r$.\n"
                   "Client knows: $m$, its limit $r$, the prior on $c$,\n"
                   "the dealer's rule (step 3); in the band\nconditions, also the band.",
        fontsize=6.6, va="top", ha="left", color="0.2", linespacing=1.35)

# ---- notation key
key = fig.add_axes([0.66, 0.0, 0.34, 1.0]); key.axis("off")
key.set_xlim(0, 1); key.set_ylim(0, 1)
key.text(0.0, 0.97, "Notation", fontsize=7.5, fontweight="bold", va="top")
rows = [("$m$", "evaluated price: an indicative price both sides see"),
        ("$c$", "dealer's cost; private to the dealer"),
        ("$r$", "client's limit, the most it will pay; private to the client"),
        ("$p$", "dealer's quote"),
        ("$q$", "client's counteroffer"),
        ("$g$", "price unit: 1/10 of daily price volatility (about \\$3,400 on \\$10mm)"),
        ("band", "disclosed range for the dealer's cost, from 10 other dealers' trades")]
y = 0.84
renderer = fig.canvas.get_renderer()
for sym, desc in rows:                         # each row starts a fixed gap below the previous one
    key.text(0.0, y, sym, fontsize=7, va="top")
    t = key.text(0.13, y, textwrap.fill(desc, 46), fontsize=6.6, va="top", color="0.2", linespacing=1.2)
    bb = t.get_window_extent(renderer).transformed(key.transAxes.inverted())
    y = bb.y0 - 0.055
key.plot([-0.04, -0.04], [0.02, 0.98], color="0.8", lw=0.6)
save(fig, "fig1_setup")
plt.show()


## Figure 2: does the band help?

One column. A slope chart of the primary comparison: each client's share of the surplus without and with the band (no cost question for the LLM). The rational client's line rises; the LLM's does not.

In [ ]:
fig, ax = plt.subplots(figsize=(COL, 2.1))
X = [0, 1]
llm = [capture_by_arm["U_noelicit"], capture_by_arm["B_noelicit"]]
rat = [rational["U"], rational["B"]]
llm_eff, rat_eff = effect_rows[0][1], effect_rows[0][3]
llm_ci = effect_rows[0][2]

for vals, col, mfc, name in ((rat, RATIONAL, "white", "Rational client"),
                             (llm, BAND, BAND, "LLM client")):
    m = [v[0] for v in vals]
    err = [[v[0] - v[1][0] for v in vals], [v[1][1] - v[0] for v in vals]]
    ax.plot(X, m, color=col, lw=1.2, zorder=2)
    ax.errorbar(X, m, yerr=err, fmt="o", color=col, mfc=mfc, mec=col, mew=1.0, ms=4.5, lw=1.0, zorder=3)
    for x, v, ha, dx in ((0, m[0], "right", -0.06), (1, m[1], "left", 0.06)):
        ax.text(x + dx, v, f"{v:.2f}", ha=ha, va="center", fontsize=7)

ax.text(1.24, rat[1][0], f"Rational client\n(+{rat_eff:.2f})", va="center", ha="left", fontsize=7)
minus = lambda t: t.replace("-", "−")                 # typographic minus sign
ax.text(1.24, llm[1][0], minus(f"LLM client\n({llm_eff:+.2f}, 95% CI\n{llm_ci[0]:+.2f} to {llm_ci[1]:+.2f})"),
        va="center", ha="left", fontsize=7)

ax.set_xticks(X)
ax.set_xticklabels(["Without band", "With band"])
ax.set_xlim(-0.35, 1.95)
ax.set_ylim(0.2, 0.85)
ax.yaxis.set_major_locator(MultipleLocator(0.1))
ax.set_ylabel("client capture")
ax.spines["bottom"].set_bounds(0, 1)
save(fig, "fig2_results")
plt.show()

## Figure 3: where the client counters

One column, the no-question conditions (the primary comparison). (a) Counteroffers relative to the evaluated price, with the briefing's numbers marked. (b) The same counteroffers relative to the band's centre, with its stated range shaded.

In [ ]:
def counters(recs, cond, ref):
    """Stated counteroffers in g, relative to m (ref='m') or to the band's centre (ref='avg')."""
    out = []
    for r in recs:
        if r["key"]["condition"] == cond and r["decision"]["action"] == "counter":
            base = r["scenario"]["m"] if ref == "m" else r["band"]["avg"]
            out.append((r["decision"]["counter_stated"] - base) / G)
    return np.array(out)

WIDTH = 0.05
fig, (a1, a2) = plt.subplots(2, 1, figsize=(COL, 2.75), gridspec_kw=dict(hspace=0.75))

for ax, ref, lim in ((a1, "m", (-2.6, 0.3)), (a2, "avg", (-2.0, 2.0))):
    bins = np.arange(lim[0], lim[1] + WIDTH, WIDTH)
    nb, bb = counters(MAIN, "U_noelicit", ref), counters(MAIN, "B_noelicit", ref)
    ax.hist(nb, bins=bins, color="0.78", lw=0, label=f"no band ($n$ = {len(nb)})")
    ax.hist(bb, bins=bins, histtype="step", color=BAND, lw=1.0, label=f"band ($n$ = {len(bb)})")
    ax.set_xlim(*lim)
    ax.set_ylabel("counteroffers")

ymax = a1.get_ylim()[1] * 1.15
a1.set_ylim(0, ymax)
for v, t in ((-1.0, "briefing's\ntypical cost"), (-1.898, "briefing's\n80% range edge")):
    a1.axvline(v, color="0.25", lw=0.6, ls=":")
    a1.text(v - 0.04, ymax * 0.97, t, fontsize=6.3, va="top", ha="right", color="0.25")
a1.set_xlabel("counteroffer relative to the evaluated price $m$ (units of $g$)")
a1.legend(loc="upper right", borderaxespad=0.0, bbox_to_anchor=(1.0, 1.0))
panel(a1, "a", x=-0.14)

ymax = a2.get_ylim()[1] * 1.5
a2.set_ylim(0, ymax)
a2.axvspan(-S_BAND, S_BAND, color=BAND_LIGHT, lw=0, zorder=0)
a2.axvline(0, color=BAND, lw=0.6, ls=":")
a2.text(-S_BAND - 0.04, ymax * 0.97, "band's stated\nrange", fontsize=6.3, va="top", ha="right", color="0.25")
a2.text(0.04, ymax * 0.97, "band\ncentre", fontsize=6.3, va="top", ha="left", color="0.25")
a2.set_xlabel("counteroffer relative to the band's centre (units of $g$)")
panel(a2, "b", x=-0.14)
save(fig, "fig3_counters")
plt.show()

## Figure 4: how far each piece of evidence moves the client's best guess

One column, conditions with belief elicitation. Quote row: size of the move in the best guess (median of the stated belief) from before to after the quote, without the band; the Bayesian client here inverts the dealer's quoting rule exactly. Band row: how much closer the pre-quote best guess sits to the band's centre with the band than without (pre-registered uptake). Means over 50 scenarios with 95% bootstrap CIs.

In [ ]:
from analysis import stated_cdf, scenario_of

# ---------------------------------------------------------------- Figure 4: how far each piece of evidence moves the best guess
# Two pieces of evidence about the dealer's cost: its quote (reveals little) and the band (reveals a lot).
# For each, how far does the client's best guess (median of its belief, in g) move towards it?
#   Quote, LLM:      size of the move from pre- to post-quote best guess, conditions without the band (U).
#   Quote, Bayesian: a client that knows the dealer's quoting rule and inverts it exactly
#                    (prior on c, conditioned on "this cost would have produced this quote").
#   Band, LLM:       fall in distance from the pre-quote best guess to the band's centre, B vs U (pre-registered uptake).
#   Band, Bayesian:  the same for the Bayesian posterior given the band.


def median_g(cdf, q, m):
    return (float(np.interp(0.5, cdf + np.arange(len(cdf)) * 1e-12, q)) - m) / G    # tiny ramp keeps cdf increasing


def llm_guess(r, which):
    scn = scenario_of(r)
    q = price_grid(scn.m, G)
    return median_g(stated_cdf(r["beliefs"][which], scn, G, q), q, scn.m)


def rounded_quote(m, c):
    """The dealer's quote exactly as in the experiment: p* to the nearest 0.001, never below cost."""
    p = round(dealer_best_quote(m, c, G), 3)
    return round(p + 0.001, 3) if p < c else p


def bayes_quote_shift(m, p):
    """Move in the best guess of a client that inverts the dealer's quoting rule exactly (in g)."""
    q = price_grid(m, G)
    cs = q[q < m]                                            # possible costs
    prior = prior_density_c(cs, m, G)
    same = np.array([rounded_quote(m, c) == p for c in cs])  # costs that would have produced this quote
    post = prior * same
    if post.sum() == 0:
        return 0.0
    cdf_prior, cdf_post = np.cumsum(prior) / prior.sum(), np.cumsum(post) / post.sum()
    return median_g(cdf_post, cs, m) - median_g(cdf_prior, cs, m)


def bayes_band_guess(avg_g, shown):
    q = price_grid(100.0, G)
    if shown:
        _, cdf = band_posterior(q, 100.0, G, 100.0 + avg_g * G, S_BAND * G)
    else:
        d = prior_density_c(q, 100.0, G)
        cdf = np.cumsum(d) / d.sum()
    return median_g(cdf, q, 100.0)


def by_scenario(pairs):
    out = {}
    for s, v in pairs:
        out.setdefault(s, []).append(v)
    return {s: float(np.mean(v)) for s, v in out.items()}


elicited = lambda cond: [r for r in MAIN if r["key"]["condition"] == cond
                         and r["beliefs"]["pre"] is not None and r["beliefs"]["post"] is not None]

# quote: how far the best guess moves after seeing the quote (size of the move, either direction)
llm_q = by_scenario((r["key"]["seed"], abs(llm_guess(r, "post") - llm_guess(r, "pre"))) for r in elicited("U"))
bay_q = {}
for r in elicited("U"):
    s = r["key"]["seed"]
    if s not in bay_q:
        bay_q[s] = abs(bayes_quote_shift(r["scenario"]["m"], r["episode"]["quote"]))

# band: how much closer the pre-quote best guess sits to the band's centre with the band than without
def band_dist(cond, who):
    pairs = []
    for r in elicited(cond):
        m = r["scenario"]["m"]
        a = (r["band"]["avg"] - m) / G
        guess = llm_guess(r, "pre") if who == "LLM" else bayes_band_guess(a, cond == "B")
        pairs.append((r["key"]["seed"], abs(guess - a)))
    return by_scenario(pairs)

shift = {}
for who in ("LLM", "Bayes"):
    u, b = band_dist("U", who), band_dist("B", who)
    shift[("band", who)] = [u[s] - b[s] for s in sorted(set(u) & set(b))]
shift[("quote", "LLM")] = list(llm_q.values())
shift[("quote", "Bayes")] = list(bay_q.values())

est = {k: bootstrap_ci(v) for k, v in shift.items()}
for k, (m_, (lo, hi)) in est.items():
    print(f"{k[0]:<5} {k[1]:<5} moves best guess {m_:+.2f}g  [{lo:+.2f}, {hi:+.2f}]  (n = {len(shift[k])} scenarios)")

# ---- plot
fig, ax = plt.subplots(figsize=(COL, 1.55))
rows = [("quote", "Dealer's quote\n(reveals little)", 1.0), ("band", "Band\n(reveals a lot)", 0.0)]
for key, label, y in rows:
    for who, col, mfc, dy in (("Bayes", RATIONAL, "white", 0.17), ("LLM", BAND, BAND, -0.17)):
        m_, (lo, hi) = est[(key, who)]
        ax.errorbar(m_, y + dy, xerr=[[m_ - lo], [hi - m_]], fmt="o", color=col, mfc=mfc, mec=col,
                    mew=1.0, ms=4.5, lw=1.0, zorder=3)
        ax.text(hi + 0.025, y + dy, f"{m_:.2f}", va="center", ha="left", fontsize=6.8)
ax.set_yticks([y for _, _, y in rows])
ax.set_yticklabels([lab for _, lab, _ in rows])
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.set_ylim(-0.5, 1.75)
ax.set_xlim(0, 0.8)
ax.xaxis.set_major_locator(MultipleLocator(0.2))
ax.set_xlabel("how far the evidence moved the best guess (units of $g$)")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", color=RATIONAL, mfc="white", lw=0, ms=4.5, label="Bayesian client"),
                   Line2D([], [], marker="o", color=BAND, mfc=BAND, lw=0, ms=4.5, label="LLM client")],
          loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=2, handletextpad=0.3, columnspacing=1.5)
save(fig, "fig4_evidence")
plt.show()


## Figure 5: main experiment against held-out scenarios

One column. (a) The band's effect on capture (H1), (b) bids moved towards the band's centre (A1), (c) the best guess moved towards the band's centre (uptake), main against held-out, with the Bayesian client where it has an analogue. All LLM numbers use the pre-registered functions in `disclosure_results.py`.

In [ ]:
from disclosure_results import dist_to_band, belief_median_to_band

# ---------------------------------------------------------------- Figure 5: main experiment against held-out scenarios
# Every LLM number comes from the pre-registered functions in disclosure_results.py (paired, bootstrap over scenarios):
#   (a) H1: capture, B_noelicit - U_noelicit, in percentage points; Bayesian: mean capture(B*) - capture(U*).
#   (b) A1: fall in |counter - band centre|, U_noelicit minus B_noelicit, in g (no Bayesian analogue).
#   (c) Uptake: fall in |best guess - band centre|, U minus B, in g; Bayesian: same for its posterior median.


def bayes_guess_g(avg_g, shown):
    """Median of a Bayesian client's belief about the cost (prior, or posterior given the band), in g from m."""
    q = price_grid(100.0, G)
    if shown:
        _, cdf = band_posterior(q, 100.0, G, 100.0 + avg_g * G, S_BAND * G)
    else:
        d = prior_density_c(q, 100.0, G)
        cdf = np.cumsum(d) / d.sum()
    return (float(np.interp(0.5, cdf, q)) - 100.0) / G


def bayes_uptake(recs, seeds):
    """Mean over scenarios of |prior median - avg| - |posterior median - avg|, in g."""
    vals = []
    for s in seeds:
        r = next(x for x in recs if x["key"]["seed"] == s)
        a = (r["band"]["avg"] - r["scenario"]["m"]) / G
        vals.append(abs(bayes_guess_g(a, False) - a) - abs(bayes_guess_g(a, True) - a))
    return float(np.mean(vals))


sets = {"Main": (MAIN, SEEDS_MAIN, bench_main), "Held-out": (HELDOUT, SEEDS_HELD, bench_held)}
res = {}
for name, (recs, seeds, bench) in sets.items():
    h1 = paired(recs, "B_noelicit", "U_noelicit", capture, seeds)
    a1 = paired(recs, "B_noelicit", "U_noelicit", lambda r: dist_to_band(r, G), seeds)
    up = paired(recs, "B", "U", lambda r: belief_median_to_band(r, G), seeds)
    res[name] = {
        "a": (100 * h1["diff"], [100 * x for x in h1["ci"]], 100 * rational_gain(bench)),
        "b": (-a1["diff"], [-x for x in a1["ci"][::-1]], None),
        "c": (-up["diff"], [-x for x in up["ci"][::-1]], bayes_uptake(recs, seeds)),
    }
    for k, (m_, ci, bay) in res[name].items():
        extra = "" if bay is None else f"   Bayesian {bay:+.2f}"
        print(f"{name:<9} ({k}) LLM {m_:+.2f} [{ci[0]:+.2f}, {ci[1]:+.2f}]{extra}")

fig, axes = plt.subplots(3, 1, figsize=(COL, 3.2), gridspec_kw=dict(hspace=0.95))
labels = {"a": "band effect on capture (percentage points)",
          "b": "bids moved towards the band's centre ($g$)",
          "c": "best guess moved towards the band's centre ($g$)"}
lims = {"a": (-25, 25), "b": (-0.1, 0.6), "c": (-0.3, 0.7)}
for ax, k in zip(axes, "abc"):
    for y, name in ((1, "Main"), (0, "Held-out")):
        m_, ci, bay = res[name][k]
        mfc = BAND if name == "Main" else "white"
        ax.errorbar(m_, y, xerr=[[m_ - ci[0]], [ci[1] - m_]], fmt="o", color=BAND, mfc=mfc, mec=BAND,
                    mew=1.0, ms=4.5, lw=1.0, zorder=3)
        if bay is not None:
            ax.plot(bay, y, "D", ms=4.5, mfc="white", mec=RATIONAL, mew=1.0, zorder=4)
    ax.axvline(0, color="0.4", lw=0.6, zorder=1)
    ax.set_yticks([1, 0])
    ax.set_yticklabels(["Main", "Held-out"])
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.set_ylim(-0.6, 1.6)
    ax.set_xlim(*lims[k])
    ax.set_xlabel(labels[k])
    panel(ax, k, x=-0.22, y=0.95)
from matplotlib.lines import Line2D
axes[0].legend(handles=[Line2D([], [], marker="o", color=BAND, mfc=BAND, lw=0, ms=4.5, label="LLM client (95% CI)"),
                        Line2D([], [], marker="D", color=RATIONAL, mfc="white", lw=0, ms=4.5, label="Bayesian client")],
               loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, handletextpad=0.3, columnspacing=1.5)
save(fig, "fig5_heldout")
plt.show()


## Numbers quoted in the text

The exploratory finding in Section 4.4: how often counteroffers in the no-question conditions sit within one tick (0.001 points) of the band's bounds, and how those counteroffers fare.

In [ ]:
TICK = 0.0011
for cond in ("U_noelicit", "B_noelicit"):
    rs = [r for r in MAIN if r["key"]["condition"] == cond and r["decision"]["action"] == "counter"]
    for key in ("low", "avg", "high"):
        sel = [r for r in rs if abs(r["decision"]["counter_stated"] - r["band"][key]) < TICK]
        rej = np.mean([r["episode"]["status"] == "counter_rejected" for r in sel]) if sel else float("nan")
        print(f"{cond:<11} at band {key:<4}: {len(sel):2d} of {len(rs)} ({len(sel) / len(rs):.0%}), rejected {rej:.0%}")

## Exploratory numbers quoted in the text

Not pre-registered. Stated probabilities before and after the quote, and whether bids follow beliefs (with the band, also after removing the band centre's common pull).

In [ ]:
from analysis import own_best_counter, scenario_of
from render import BELIEF_GAPS

# ---------------------------------------------------------------- Exploratory numbers quoted in the text (Section 4.5)

# 1. Stated probabilities before and after the quote (conditions with elicitation), e.g. at 1g below m: 0.71 -> 0.36
for cond in ("U", "B"):
    rs = [r for r in MAIN if r["key"]["condition"] == cond and r["beliefs"]["pre"] and r["beliefs"]["post"]]
    pre = np.array([r["beliefs"]["pre"] for r in rs]).mean(0)
    post = np.array([r["beliefs"]["post"] for r in rs]).mean(0)
    print(f"{cond}: P(cost at least x below m), before -> after the quote, mean drop {np.mean(pre - post):.3f}")
    for gap, a, b in zip(BELIEF_GAPS, pre, post):
        print(f"   {gap:.2f}g: {a:.2f} -> {b:.2f}")


# 2. Do bids follow beliefs? Rank correlation of the actual counter with the counter a rational client holding the
#    stated post-quote belief would make; with the band, also after removing the band centre's common pull.
def rank_resid(y, x):
    y, x = stats.rankdata(y), stats.rankdata(x)
    return y - np.polyval(np.polyfit(x, y, 1), x)


for cond in ("U", "B"):
    actual, implied, centre = [], [], []
    for r in MAIN:
        if r["key"]["condition"] == cond and r["decision"]["action"] == "counter" and r["beliefs"]["post"]:
            scn = scenario_of(r)
            actual.append((r["decision"]["counter_stated"] - scn.m) / G)
            implied.append((own_best_counter(r["beliefs"]["post"], scn, G) - scn.m) / G)
            centre.append((r["band"]["avg"] - scn.m) / G)
    rho, p = stats.spearmanr(implied, actual)
    pr, pp = stats.pearsonr(rank_resid(actual, centre), rank_resid(implied, centre))
    print(f"{cond}: n {len(actual)}, rank correlation {rho:.2f} (p {p:.3f}); "
          f"controlling for the band centre {pr:.2f} (p {pp:.3f})")
